# Real jira

In [ ]:
"""
==============================================================================
  Real-World Ground Truth Builder — Task Design, Task Allocation,
  Sprint Planning, Resource Planning
  ------------------------------------------------------------------------
  One consistent real-data source (Apache Jira) feeding all four of your
  ipynb pipelines. Each function below produces samples in the SAME
  structural format your synthetic notebooks expect (task_type, input,
  reference_output) but with real fields substituted wherever real ground
  truth actually exists.

  IMPORTANT ASF-specific adaptation:
  Apache projects are volunteer-run and mostly don't use Jira's Scrum
  "Sprint" feature (confirmed: the Agile board/sprint API isn't reliably
  open for anonymous access on issues.apache.org). The real, available
  analog is `fixVersion` (a release) — it has a real scope, a real cutoff,
  and real issues that got pushed out. This script uses fixVersion as the
  real substitute for "sprint" everywhere Sprint Planning/Resource
  Planning would otherwise need a Scrum sprint. State this substitution
  explicitly in your Methodology section — it's a deliberate, documented
  adaptation, not a hidden shortcut.
==============================================================================
"""

import requests, time, json
from collections import defaultdict
from pathlib import Path

ASF_JIRA = "https://issues.apache.org/jira/rest/api/2/search"


def _search(jql: str, fields: str, max_results: int = 300) -> list:
    params = {"jql": jql, "maxResults": max_results, "fields": fields}
    r = requests.get(ASF_JIRA, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("issues", [])


def priority_to_scale(p: str) -> str:
    p = (p or "").lower()
    return {"blocker": "Critical", "critical": "Critical", "major": "High",
            "minor": "Medium", "trivial": "Low"}.get(p, "Medium")


# ---------------------------------------------------------------------------
# 1. TASK DESIGN — real issue -> structured ticket (reference-free + priority check)
# ---------------------------------------------------------------------------
TASK_DESIGN_PROMPT = """You are a project manager converting a real engineering issue into a structured Jira-style ticket.

ISSUE TITLE: {title}
ISSUE DESCRIPTION:
{description}

Return JSON: {{"title","description","acceptance_criteria":[...],"story_points":<num>,"priority","labels":[...],"definition_of_done"}}"""


def build_task_design(project: str, n: int = 60) -> list:
    issues = _search(
        f'project = {project} AND resolution = Fixed AND description is not EMPTY ORDER BY updated DESC',
        "summary,description,priority,labels", max_results=n,
    )
    samples = []
    for i in issues:
        f = i["fields"]
        if not f.get("description"):
            continue
        samples.append({
            "id": f"real_design_{i['key']}", "task_type": "task_design",
            "input": TASK_DESIGN_PROMPT.format(title=f["summary"], description=f["description"][:1200]),
            "reference_output": "",  # no synthetic reference — schema + judge + priority-check only
            "is_real": True, "source_url": f"https://issues.apache.org/jira/browse/{i['key']}",
            "known_priority": priority_to_scale((f.get("priority") or {}).get("name")),
        })
    return samples


# ---------------------------------------------------------------------------
# 2. TASK ALLOCATION — real assignee + real logged hours as ground truth
# ---------------------------------------------------------------------------
def build_task_allocation(project: str, n_eval: int = 40) -> list:
    issues = _search(
        f'project = {project} AND resolution = Fixed AND assignee is not EMPTY '
        f'AND timespent is not EMPTY ORDER BY resolutiondate DESC',
        "summary,assignee,components,timespent,timeoriginalestimate", max_results=300,
    )
    if len(issues) < n_eval + 20:
        return []
    history, evalset = issues[n_eval:], issues[:n_eval]

    skills = defaultdict(lambda: defaultdict(int))
    for i in history:
        f = i["fields"]
        who = (f.get("assignee") or {}).get("displayName")
        if not who:
            continue
        for c in f.get("components", []):
            skills[who][c["name"]] += 1
    roster = {name: sorted(cc, key=cc.get, reverse=True)[:4]
              for name, cc in skills.items() if sum(cc.values()) >= 3}
    if len(roster) < 3:
        return []
    team = [{"name": n_, "role": f"{s[0]} Engineer" if s else "Engineer", "skills": s, "capacity": 8}
            for n_, s in roster.items()]

    samples = []
    for i in evalset:
        f = i["fields"]
        who = (f.get("assignee") or {}).get("displayName")
        if who not in roster:
            continue
        secs = f.get("timespent") or f.get("timeoriginalestimate")
        if not secs:
            continue
        comp = (f.get("components") or [{"name": "General"}])[0]["name"]
        samples.append({
            "id": f"real_alloc_{i['key']}", "task_type": "task_allocation",
            "input": (f"You are a project manager assigning tasks to team members.\n\n"
                      f"TASK: {f['summary']}\nREQUIRED COMPONENT: {comp}\nCOMPLEXITY: medium\n\n"
                      f"AVAILABLE TEAM:\n{json.dumps(team, indent=2)}\n\n"
                      f'Return JSON: {{"assigned_to","role","rationale","skill_match":[...],'
                      f'"estimated_hours":<num>,"pairing_suggestion","risks"}}'),
            "reference_output": json.dumps({"assigned_to": who, "estimated_hours": round(secs/3600, 1)}),
            "is_real": True, "source_url": f"https://issues.apache.org/jira/browse/{i['key']}",
            "ground_truth_assignee": who, "ground_truth_hours": round(secs/3600, 1),
        })
    return samples


# ---------------------------------------------------------------------------
# 3. SPRINT PLANNING — real fixVersion (release) as the "sprint"
# ---------------------------------------------------------------------------
STORY_POINTS_FIELD = "customfield_12310243"  # ASF's actual story points field id — verify per-project via /rest/api/2/field

def build_sprint_planning(project: str, n_versions: int = 6) -> list:
    # pull recent released fixVersions with resolved issues
    issues = _search(
        f'project = {project} AND fixVersion is not EMPTY AND resolution = Fixed ORDER BY resolutiondate DESC',
        f"summary,fixVersions,{STORY_POINTS_FIELD},resolutiondate", max_results=400,
    )
    by_version = defaultdict(list)
    for i in issues:
        for v in i["fields"].get("fixVersions", []):
            by_version[v["name"]].append(i)

    # also pull issues that were open around the same time but NOT in that version (candidates "excluded")
    samples = []
    versions = sorted(by_version, key=lambda v: len(by_version[v]), reverse=True)[:n_versions]
    for v in versions:
        included = by_version[v]
        if len(included) < 4:
            continue
        points = [i["fields"].get(STORY_POINTS_FIELD) for i in included]
        points = [p for p in points if isinstance(p, (int, float))]
        total_points = sum(points) if points else None

        backlog_candidates = _search(
            f'project = {project} AND fixVersion != "{v}" AND status = Open ORDER BY priority DESC',
            "summary", max_results=10,
        )

        samples.append({
            "id": f"real_sprint_{project}_{v}".replace(" ", "_"),
            "task_type": "sprint_planning",
            "input": (
                f"You are planning release '{v}' for project {project}.\n\n"
                f"BACKLOG CANDIDATES:\n" +
                "\n".join(f"- {i['fields']['summary']}" for i in (included + backlog_candidates)) +
                "\n\nSelect the tasks for this release, estimate total story points, "
                "and state exclusion reasons for anything left out.\n\n"
                'Return JSON: {"sprint_goal","selected_tasks":[...],"total_points":<num>,"excluded_tasks":[...]}'
            ),
            "reference_output": json.dumps({
                "selected_tasks": [i["fields"]["summary"] for i in included],
                "total_points": total_points,
                "excluded_tasks": [i["fields"]["summary"] for i in backlog_candidates],
            }),
            "is_real": True, "note": "sprint = real ASF fixVersion (release), not a Scrum sprint",
        })
    return samples


# ---------------------------------------------------------------------------
# 4. RESOURCE PLANNING — real velocity (points/time across releases) + real slippage as risk signal
# ---------------------------------------------------------------------------
def build_resource_planning(project: str) -> list:
    issues = _search(
        f'project = {project} AND resolution = Fixed AND fixVersion is not EMPTY ORDER BY resolutiondate DESC',
        f"fixVersions,{STORY_POINTS_FIELD},resolutiondate", max_results=500,
    )
    by_version = defaultdict(lambda: {"points": 0, "count": 0, "dates": []})
    for i in issues:
        pts = i["fields"].get(STORY_POINTS_FIELD)
        for v in i["fields"].get("fixVersions", []):
            by_version[v["name"]]["points"] += pts if isinstance(pts, (int, float)) else 0
            by_version[v["name"]]["count"] += 1
            if i["fields"].get("resolutiondate"):
                by_version[v["name"]]["dates"].append(i["fields"]["resolutiondate"])

    # real velocity = average points delivered per release over the last few releases
    recent = list(by_version.items())[:5]
    if len(recent) < 3:
        return []
    avg_velocity = round(sum(v["points"] for _, v in recent) / len(recent), 1)

    # real risk signal: issues whose target fixVersion changed (i.e. they slipped) — proxy via
    # issues still Open/In-Progress with a fixVersion already in the past (heuristic slippage)
    slipped = _search(
        f'project = {project} AND fixVersion is not EMPTY AND status not in (Resolved, Closed) '
        f'ORDER BY updated ASC',
        "summary,fixVersions", max_results=15,
    )

    return [{
        "id": f"real_resource_{project}",
        "task_type": "resource_planning",
        "input": (
            f"Project {project} has an observed real velocity of {avg_velocity} story points "
            f"per release (avg over last {len(recent)} releases).\n"
            f"CURRENTLY AT-RISK ITEMS (past target, still open):\n" +
            "\n".join(f"- {i['fields']['summary']}" for i in slipped) +
            "\n\nGiven this capacity, plan resource allocation for the next release: "
            "identify capacity constraints, resource risk flags, and mitigation recommendations.\n\n"
            'Return JSON: {"capacity_constraints","risk_flags":[...],"mitigation_recommendations":[...]}'
        ),
        "reference_output": json.dumps({
            "real_avg_velocity": avg_velocity,
            "real_slipped_items": [i["fields"]["summary"] for i in slipped],
        }),
        "is_real": True,
        "note": "velocity/risk derived from real release history, not synthetic",
    }]


def main():
    projects = ["SPARK", "KAFKA", "HADOOP", "HBASE", "FLINK", "CASSANDRA"]
    out = {"task_design": [], "task_allocation": [], "sprint_planning": [], "resource_planning": []}
    for p in projects:
        print(f"=== {p} ===")
        out["task_design"]      += build_task_design(p)
        out["task_allocation"]  += build_task_allocation(p)
        out["sprint_planning"]  += build_sprint_planning(p)
        out["resource_planning"]+= build_resource_planning(p)
        time.sleep(1)

    Path("dataset").mkdir(exist_ok=True)
    for task_type, samples in out.items():
        path = Path(f"dataset/real_{task_type}.json")
        path.write_text(json.dumps(samples, indent=2))
        print(f"{task_type}: {len(samples)} samples -> {path}")


if __name__ == "__main__":
    main()

=== SPARK ===
=== KAFKA ===
=== HADOOP ===
=== HBASE ===
=== FLINK ===
=== CASSANDRA ===
task_design: 360 samples -> dataset/real_task_design.json
task_allocation: 52 samples -> dataset/real_task_allocation.json
sprint_planning: 36 samples -> dataset/real_sprint_planning.json
resource_planning: 6 samples -> dataset/real_resource_planning.json


### 50 more samples

In [ ]:
"""
==============================================================================
  Fetch 50+ real Sprint Planning + Resource Planning samples from Apache Jira
  ------------------------------------------------------------------------
  Self-contained — run this directly (needs open internet, e.g. Colab).
  Widens project/version coverage vs the earlier run (which capped at 36)
  and derives resource_planning per-release (not per-project), so both
  tracks scale together automatically.
==============================================================================
"""

import requests, time, json
from collections import defaultdict
from pathlib import Path

ASF_JIRA = "https://issues.apache.org/jira/rest/api/2/search"

PROJECTS = [
    "SPARK", "KAFKA", "HADOOP", "HBASE", "FLINK", "CASSANDRA",
    "AIRFLOW", "BEAM", "NIFI", "ZOOKEEPER", "HIVE", "OOZIE",
]
N_VERSIONS_PER_PROJECT = 12  # was 6 — main lever for hitting 50+


def _search(jql: str, fields: str, max_results: int = 400) -> list:
    params = {"jql": jql, "maxResults": max_results, "fields": fields}
    r = requests.get(ASF_JIRA, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("issues", [])


def build_sprint_planning(project: str, n_versions: int = N_VERSIONS_PER_PROJECT) -> list:
    issues = _search(
        f'project = {project} AND fixVersion is not EMPTY AND resolution = Fixed '
        f'ORDER BY resolutiondate DESC',
        "summary,fixVersions", max_results=400,
    )
    by_version = defaultdict(list)
    for i in issues:
        for v in i["fields"].get("fixVersions", []):
            by_version[v["name"]].append(i)

    samples = []
    versions = sorted(by_version, key=lambda v: len(by_version[v]), reverse=True)[:n_versions]
    for v in versions:
        included = by_version[v]
        if len(included) < 4:
            continue

        backlog_candidates = _search(
            f'project = {project} AND fixVersion != "{v}" AND status = Open ORDER BY priority DESC',
            "summary", max_results=10,
        )

        samples.append({
            "id": f"real_sprint_{project}_{v}".replace(" ", "_"),
            "task_type": "sprint_planning",
            "input": (
                f"You are planning release '{v}' for project {project}.\n\n"
                f"BACKLOG CANDIDATES:\n" +
                "\n".join(f"- {i['fields']['summary']}" for i in (included + backlog_candidates)) +
                "\n\nSelect the tasks for this release, estimate total scope, "
                "and state exclusion reasons for anything left out.\n\n"
                'Return JSON: {"sprint_goal","selected_tasks":[...],"scope_size":<num>,"excluded_tasks":[...]}'
            ),
            "reference_output": json.dumps({
                "selected_tasks": [i["fields"]["summary"] for i in included],
                "scope_size": len(included),
                "excluded_tasks": [i["fields"]["summary"] for i in backlog_candidates],
                "note": "story_points unavailable on this ASF project; scope_size (issue count) used as real metric",
            }),
            "is_real": True, "project": project,
        })
        time.sleep(0.3)  # be polite to ASF infra
    return samples


def build_resource_planning_per_release(sprint_samples: list) -> list:
    """Derive one resource_planning sample per release, using prior-release
    scope as real velocity history and this release's excluded_tasks as
    real deferred/at-risk items. No extra API calls needed."""
    by_project = defaultdict(list)
    for s in sprint_samples:
        by_project[s["project"]].append(s)

    resource_samples = []
    for project, releases in by_project.items():
        scopes = [json.loads(r["reference_output"])["scope_size"] for r in releases]
        for i, r in enumerate(releases):
            ref = json.loads(r["reference_output"])
            prior = scopes[:i] if i > 0 else scopes
            avg_velocity = round(sum(prior) / len(prior), 1) if prior else None
            version = r["id"].split(f"real_sprint_{project}_")[1]

            resource_samples.append({
                "id": f"real_resource_{project}_{version}",
                "task_type": "resource_planning",
                "input": (
                    f"Project {project} has an observed real velocity of {avg_velocity} "
                    f"issues resolved per release (avg over prior releases).\n"
                    f"ITEMS EXCLUDED FROM the '{version}' release (real at-risk/deferred items):\n" +
                    "\n".join(f"- {t}" for t in ref["excluded_tasks"]) +
                    "\n\nGiven this capacity, plan resource allocation for the next release: "
                    "identify capacity constraints, resource risk flags, and mitigation "
                    "recommendations.\n\nReturn JSON: "
                    '{"capacity_constraints","risk_flags":[...],"mitigation_recommendations":[...]}'
                ),
                "reference_output": json.dumps({
                    "real_avg_velocity": avg_velocity,
                    "real_deferred_items": ref["excluded_tasks"],
                    "real_scope_this_release": ref["scope_size"],
                }),
                "is_real": True, "project": project,
            })
    return resource_samples


def main():
    all_sprint = []
    for p in PROJECTS:
        print(f"Fetching {p} ...")
        try:
            s = build_sprint_planning(p)
            print(f"  -> {len(s)} releases usable")
            all_sprint.extend(s)
        except Exception as e:
            print(f"  failed: {e}")
        time.sleep(1)

    all_resource = build_resource_planning_per_release(all_sprint)

    Path("dataset").mkdir(exist_ok=True)
    Path("dataset/real_sprint_planning_v2.json").write_text(json.dumps(all_sprint, indent=2))
    Path("dataset/real_resource_planning_v2.json").write_text(json.dumps(all_resource, indent=2))

    print(f"\nsprint_planning:   {len(all_sprint)} samples")
    print(f"resource_planning: {len(all_resource)} samples")
    if len(all_sprint) < 50 or len(all_resource) < 50:
        print("⚠️  Still under 50 — add more projects to PROJECTS or raise N_VERSIONS_PER_PROJECT")


if __name__ == "__main__":
    main()

Fetching SPARK ...
  -> 11 releases usable
Fetching KAFKA ...
  -> 9 releases usable
Fetching HADOOP ...
  -> 6 releases usable
Fetching HBASE ...
  -> 12 releases usable
Fetching FLINK ...
  -> 12 releases usable
Fetching CASSANDRA ...
  -> 12 releases usable
Fetching AIRFLOW ...
  failed: 400 Client Error:  for url: https://issues.apache.org/jira/rest/api/2/search?jql=project+%3D+AIRFLOW+AND+fixVersion+is+not+EMPTY+AND+resolution+%3D+Fixed+ORDER+BY+resolutiondate+DESC&maxResults=400&fields=summary%2CfixVersions
Fetching BEAM ...
  -> 5 releases usable
Fetching NIFI ...
  -> 7 releases usable
Fetching ZOOKEEPER ...
  -> 12 releases usable
Fetching HIVE ...
  -> 3 releases usable
Fetching OOZIE ...
  -> 7 releases usable

sprint_planning:   96 samples
resource_planning: 96 samples
